# User Config

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter, deque
import pandas as pd
import json
import random


# =========================================================
# CONFIG
# =========================================================
LABEL_PATH = Path("./datasets/label.csv")
SPLIT_PATH = Path("./datasets/split.csv")

EDGE_CSV_PATH = Path("./datasets/edge.csv")
EDGE_PARQUET_PATH = Path("./datasets/edge.parquet")

USER_JSON_PATH = Path("./datasets/user.json")

OUTPUT_DIR = Path("./datasets/final_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_USERS_FINAL = OUTPUT_DIR / "df_users_final.parquet"
OUTPUT_EDGES_FINAL = OUTPUT_DIR / "df_edges_final.parquet"

RANDOM_STATE = 42
TOTAL_N = 20_000
HUMAN_SEED_N = 7_000
BOT_SEED_N = 5_000

USE_SPLIT_FILTER = None   # e.g. "train"

# adjust follower/followers depending on actual edge file values
SOCIAL_RELATIONS = {"following", "follower"}
USER_RELEVANT_RELATIONS = {
    "followers",
    "following",
    "post",
    "pinned",
    "like",
    "own",
    "membership",
    "followed",
}
FINAL_GRAPH_RELATIONS = {
    "contain",
    "discuss",
    "followed",
    "followers",
    "following",
    "like",
    "membership",
    "mentioned",
    "own",
    "pinned",
    "post",
    "quoted",
    "replied_to",
    "retweeted",
}

# Helper Functions

In [ ]:
# =========================================================
# HELPERS
# =========================================================
def maybe_convert_csv_to_parquet(csv_path: Path, parquet_path: Path, chunksize: int = 1_000_000):
    """
    Convert a large CSV to parquet once.
    If parquet already exists, do nothing.
    """
    if parquet_path.exists():
        print(f"Parquet already exists: {parquet_path}")
        return

    print(f"Converting {csv_path} -> {parquet_path} ...")
    chunks = []
    for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunksize)):
        chunks.append(chunk)
        if i % 10 == 0:
            print(f"  read chunks: {i}")

    df = pd.concat(chunks, ignore_index=True)
    df.to_parquet(parquet_path, index=False)
    print("Conversion done.")

def load_label_split(label_path: Path, split_path: Path) -> pd.DataFrame:
    df_label = pd.read_csv(label_path)
    df_split = pd.read_csv(split_path)

    if not {"id", "label"}.issubset(df_label.columns):
        raise ValueError("label.csv must contain columns: id, label")
    if not {"id", "split"}.issubset(df_split.columns):
        raise ValueError("split.csv must contain columns: id, split")

    df_label["id"] = df_label["id"].astype(str)
    df_split["id"] = df_split["id"].astype(str)

    df_label["label"] = df_label["label"].astype(str).str.lower()
    df_split["split"] = df_split["split"].astype(str).str.lower()

    return df_label.merge(df_split, on="id", how="left")

def load_user_json_as_df(json_path: Path) -> pd.DataFrame:
    if not json_path.exists():
        raise FileNotFoundError(f"user.json not found at: {json_path}")

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, list):
            df_user = pd.json_normalize(data)
        elif isinstance(data, dict):
            list_like_keys = [k for k, v in data.items() if isinstance(v, list)]
            if len(list_like_keys) == 1:
                df_user = pd.json_normalize(data[list_like_keys[0]])
            else:
                df_user = pd.json_normalize(data)
        else:
            raise ValueError("Unsupported JSON structure.")

    except json.JSONDecodeError:
        rows = []
        with open(json_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        df_user = pd.json_normalize(rows)

    possible_id_cols = ["id", "user_id", "author_id"]
    found = None
    for col in possible_id_cols:
        if col in df_user.columns:
            found = col
            break

    if found is None:
        raise ValueError(f"No user ID column found in user.json. Columns: {df_user.columns.tolist()}")

    if found != "id":
        df_user = df_user.rename(columns={found: "id"})

    df_user["id"] = df_user["id"].astype(str)
    df_user["id"] = df_user["id"].apply(lambda x: x if x.startswith("u") else f"u{x}")
    return df_user

# Graph Views for sampling

In [16]:
def build_sampling_views_from_edge_parquet(
    edge_parquet_path: Path,
    social_relations: set[str],
    user_relevant_relations: set[str],
) -> tuple[pd.DataFrame, dict]:
    """
    Build:
    - participation degree per user
    - social adjacency user -> user
    """
    df_edges = pd.read_parquet(edge_parquet_path, columns=["source_id", "target_id", "relation"]).copy()
    df_edges["source_id"] = df_edges["source_id"].astype(str)
    df_edges["target_id"] = df_edges["target_id"].astype(str)
    df_edges["relation"] = df_edges["relation"].astype(str)

    # participation degree
    part_edges = df_edges[df_edges["relation"].isin(user_relevant_relations)].copy()

    src_users = part_edges.loc[part_edges["source_id"].str.startswith("u", na=False), "source_id"]
    tgt_users = part_edges.loc[part_edges["target_id"].str.startswith("u", na=False), "target_id"]

    participation_counter = Counter()
    participation_counter.update(src_users.tolist())
    participation_counter.update(tgt_users.tolist())

    df_degree = pd.DataFrame({
        "id": list(participation_counter.keys()),
        "participation_degree": list(participation_counter.values())
    })

    # social adjacency
    social_edges = df_edges[df_edges["relation"].isin(social_relations)].copy()
    social_edges = social_edges[
        social_edges["source_id"].str.startswith("u", na=False) &
        social_edges["target_id"].str.startswith("u", na=False)
    ]

    social_adj = defaultdict(set)
    for row in social_edges.itertuples(index=False):
        social_adj[row.source_id].add(row.target_id)
        social_adj[row.target_id].add(row.source_id)

    return df_degree, social_adj


def build_sampling_base(
    df_base: pd.DataFrame,
    df_degree: pd.DataFrame,
    split_filter: str | None = None
) -> pd.DataFrame:
    df = df_base.merge(df_degree, on="id", how="left")
    df["participation_degree"] = df["participation_degree"].fillna(0).astype(int)

    if split_filter is not None:
        df = df[df["split"] == split_filter].copy()

    df = df[df["participation_degree"] > 0].copy()
    return df

# Sampling using seed based expansion

In [18]:
def sample_balanced_seeds(
    df_sampling: pd.DataFrame,
    human_seed_n: int,
    bot_seed_n: int,
    random_state: int = 42
) -> pd.DataFrame:
    df_human = df_sampling[df_sampling["label"] == "human"].sort_values(
        "participation_degree", ascending=False
    )
    df_bot = df_sampling[df_sampling["label"] == "bot"].sort_values(
        "participation_degree", ascending=False
    )

    human_pool = df_human.head(min(len(df_human), human_seed_n * 5))
    bot_pool = df_bot.head(min(len(df_bot), bot_seed_n * 5))

    sampled_human = human_pool.sample(n=human_seed_n, random_state=random_state)
    sampled_bot = bot_pool.sample(n=bot_seed_n, random_state=random_state)

    return pd.concat([sampled_human, sampled_bot], axis=0).sample(
        frac=1, random_state=random_state
    ).reset_index(drop=True)


def seed_expand_sample(
    df_sampling: pd.DataFrame,
    social_adj: dict,
    total_n: int = 20_000,
    human_seed_n: int = 7_000,
    bot_seed_n: int = 5_000,
    random_state: int = 42
) -> pd.DataFrame:
    rng = random.Random(random_state)

    df_seed = sample_balanced_seeds(
        df_sampling=df_sampling,
        human_seed_n=human_seed_n,
        bot_seed_n=bot_seed_n,
        random_state=random_state
    )

    sampled_ids = set(df_seed["id"])
    frontier = deque(df_seed["id"].tolist())
    sampling_lookup = df_sampling.set_index("id")

    target_human = 12_200
    target_bot = 7_800

    current_human = int((df_seed["label"] == "human").sum())
    current_bot = int((df_seed["label"] == "bot").sum())

    def can_add(label: str) -> bool:
        if label == "human":
            return current_human < target_human
        if label == "bot":
            return current_bot < target_bot
        return True

    while frontier and len(sampled_ids) < total_n:
        node = frontier.popleft()
        neighbors = list(social_adj.get(node, []))
        rng.shuffle(neighbors)

        for nbr in neighbors:
            if len(sampled_ids) >= total_n:
                break
            if nbr in sampled_ids or nbr not in sampling_lookup.index:
                continue

            nbr_label = sampling_lookup.at[nbr, "label"]
            if not can_add(nbr_label):
                continue

            sampled_ids.add(nbr)
            frontier.append(nbr)

            if nbr_label == "human":
                current_human += 1
            else:
                current_bot += 1

    # fallback by highest participation degree
    if len(sampled_ids) < total_n:
        remaining = df_sampling[~df_sampling["id"].isin(sampled_ids)].sort_values(
            "participation_degree", ascending=False
        )

        for row in remaining.itertuples(index=False):
            if len(sampled_ids) >= total_n:
                break
            if row.label == "human" and current_human >= target_human:
                continue
            if row.label == "bot" and current_bot >= target_bot:
                continue

            sampled_ids.add(row.id)
            if row.label == "human":
                current_human += 1
            else:
                current_bot += 1

    if len(sampled_ids) < total_n:
        remaining = df_sampling[~df_sampling["id"].isin(sampled_ids)]
        extra = remaining.sample(
            n=min(total_n - len(sampled_ids), len(remaining)),
            random_state=random_state
        )
        sampled_ids.update(extra["id"].tolist())

    df_sampled = df_sampling[df_sampling["id"].isin(sampled_ids)].copy()

    if len(df_sampled) > total_n:
        df_sampled = df_sampled.sample(n=total_n, random_state=random_state)

    return df_sampled.reset_index(drop=True)

# Build edges file
- Start off by identifying non-user nodes connected to the 20k user sample
- For improved graph connectivity, we include an additional hop from the already identified non-user nodes

In [ ]:
def collect_first_hop_nodes(
    edge_parquet_path,
    sampled_user_ids,
):
    df_edges = pd.read_parquet(
        edge_parquet_path,
        columns=["source_id", "target_id", "relation"]
    ).copy()

    df_edges["source_id"] = df_edges["source_id"].astype(str)
    df_edges["target_id"] = df_edges["target_id"].astype(str)

    df_edges = df_edges[
        df_edges["source_id"].isin(sampled_user_ids) |
        df_edges["target_id"].isin(sampled_user_ids)
    ].copy()

    retained_users = set(sampled_user_ids)

    src = df_edges["source_id"]
    tgt = df_edges["target_id"]

    retained_tweets = set(src[src.str.startswith("t", na=False)]) | set(tgt[tgt.str.startswith("t", na=False)])
    retained_lists = set(src[src.str.startswith("l", na=False)]) | set(tgt[tgt.str.startswith("l", na=False)])
    retained_hashtags = set(src[src.str.startswith("h", na=False)]) | set(tgt[tgt.str.startswith("h", na=False)])

    return retained_users, retained_tweets, retained_lists, retained_hashtags


def expand_non_user_nodes_one_hop(
    edge_parquet_path,
    retained_users,
    retained_tweets,
    retained_lists,
    retained_hashtags,
    all_graph_relations,
):
    """
    One extra expansion pass from retained non-user nodes only.
    Users stay fixed to the sampled 20k.
    """
    df_edges = pd.read_parquet(
        edge_parquet_path,
        columns=["source_id", "target_id", "relation"]
    ).copy()

    df_edges["source_id"] = df_edges["source_id"].astype(str)
    df_edges["target_id"] = df_edges["target_id"].astype(str)
    df_edges["relation"] = df_edges["relation"].astype(str)

    df_edges = df_edges[df_edges["relation"].isin(all_graph_relations)].copy()

    retained_non_users = set(retained_tweets) | set(retained_lists) | set(retained_hashtags)

    # expand only from already retained non-user nodes
    df_edges = df_edges[
        df_edges["source_id"].isin(retained_non_users) |
        df_edges["target_id"].isin(retained_non_users)
    ].copy()

    src = df_edges["source_id"]
    tgt = df_edges["target_id"]

    # DO NOT add new users here
    retained_tweets |= set(src[src.str.startswith("t", na=False)]) | set(tgt[tgt.str.startswith("t", na=False)])
    retained_lists |= set(src[src.str.startswith("l", na=False)]) | set(tgt[tgt.str.startswith("l", na=False)])
    retained_hashtags |= set(src[src.str.startswith("h", na=False)]) | set(tgt[tgt.str.startswith("h", na=False)])

    return retained_users, retained_tweets, retained_lists, retained_hashtags


def build_edges_final_df(
    edge_parquet_path,
    retained_users,
    retained_tweets,
    retained_lists,
    retained_hashtags,
    all_graph_relations,
):
    retained_all = (
        set(retained_users)
        | set(retained_tweets)
        | set(retained_lists)
        | set(retained_hashtags)
    )

    df_edges = pd.read_parquet(
        edge_parquet_path,
        columns=["source_id", "target_id", "relation"]
    ).copy()

    df_edges["source_id"] = df_edges["source_id"].astype(str)
    df_edges["target_id"] = df_edges["target_id"].astype(str)
    df_edges["relation"] = df_edges["relation"].astype(str)

    df_edges = df_edges[df_edges["relation"].isin(all_graph_relations)].copy()
    df_edges = df_edges[
        df_edges["source_id"].isin(retained_all) &
        df_edges["target_id"].isin(retained_all)
    ].drop_duplicates().reset_index(drop=True)

    return df_edges

# Build final user metadata file

In [20]:
def build_users_final_df(
    df_sampled: pd.DataFrame,
    df_user_metadata: pd.DataFrame
) -> pd.DataFrame:
    return df_sampled.merge(df_user_metadata, on="id", how="left", suffixes=("", "_meta"))


# Main function

In [ ]:
def main():
    # 1. Convert large edge CSV to parquet once
    maybe_convert_csv_to_parquet(EDGE_CSV_PATH, EDGE_PARQUET_PATH)

    # 2. Load base user universe
    print("Loading label + split...")
    df_base = load_label_split(LABEL_PATH, SPLIT_PATH)

    # 3. Build sampling views
    print("Building sampling views...")
    df_degree, social_adj = build_sampling_views_from_edge_parquet(
        EDGE_PARQUET_PATH,
        SOCIAL_RELATIONS,
        USER_RELEVANT_RELATIONS
    )

    # 4. Build eligible sampling pool
    print("Building sampling base...")
    df_sampling = build_sampling_base(df_base, df_degree, split_filter=USE_SPLIT_FILTER)

    # 5. Sample 20k users
    print("Sampling users...")
    df_sampled = seed_expand_sample(
        df_sampling=df_sampling,
        social_adj=social_adj,
        total_n=TOTAL_N,
        human_seed_n=HUMAN_SEED_N,
        bot_seed_n=BOT_SEED_N,
        random_state=RANDOM_STATE
    )
    sampled_ids = set(df_sampled["id"])

    # 6. Load user metadata and build final users df
    print("Loading user metadata...")
    df_user_metadata = load_user_json_as_df(USER_JSON_PATH)
    df_user_metadata = df_user_metadata[df_user_metadata["id"].isin(sampled_ids)].copy()

    print("Building df_users_final...")
    df_users_final = build_users_final_df(df_sampled, df_user_metadata)

    print("Collecting first-hop nodes from sampled users...")
    retained_users, retained_tweets, retained_lists, retained_hashtags = collect_first_hop_nodes(
        EDGE_PARQUET_PATH,
        sampled_ids
    )

    print("Expanding one extra hop for non-user nodes...")
    retained_users, retained_tweets, retained_lists, retained_hashtags = expand_non_user_nodes_one_hop(
        edge_parquet_path=EDGE_PARQUET_PATH,
        retained_users=retained_users,
        retained_tweets=retained_tweets,
        retained_lists=retained_lists,
        retained_hashtags=retained_hashtags,
        all_graph_relations=FINAL_GRAPH_RELATIONS,
    )

    print("Building df_edges_final with all relation types...")
    df_edges_final = build_edges_final_df(
        edge_parquet_path=EDGE_PARQUET_PATH,
        retained_users=retained_users,
        retained_tweets=retained_tweets,
        retained_lists=retained_lists,
        retained_hashtags=retained_hashtags,
        all_graph_relations=FINAL_GRAPH_RELATIONS,
    )

    # 7. Save final outputs
    print("Saving outputs...")
    df_users_final.to_parquet(OUTPUT_USERS_FINAL, index=False)
    df_edges_final.to_parquet(OUTPUT_EDGES_FINAL, index=False)

    print("Done.")
    print(f"Users df: {OUTPUT_USERS_FINAL}")
    print(f"Edges df: {OUTPUT_EDGES_FINAL}")


if __name__ == "__main__":
    main()

Parquet already exists: datasets\edge.parquet
Loading label + split...
Building sampling views...
Building sampling base...
Sampling users...
Loading user metadata...
Building df_users_final...
Expanding one extra hop for non-user nodes...
Building df_edges_final with all relation types...
Loading tweets...
Building df_user_tweets_final...
Saving outputs...
Done.
Users df: datasets\final_outputs\df_users_final.parquet
User tweets df: datasets\final_outputs\df_user_tweets_final.parquet
Edges df: datasets\final_outputs\df_edges_final.parquet


# Build Tweets File


In [4]:
df_users_final = pd.read_parquet("./datasets/final_outputs/df_users_final.parquet")

In [ ]:
import glob
import ijson
import ijson.common
from csv import DictWriter
from pathlib import Path

# --------------------------------------------------
# 1. Build sampled user-id lookup set
# --------------------------------------------------
user_ids = set(df_users_final["id"].astype(str))
print(f"Unique sampled user IDs: {len(user_ids):,}")

# --------------------------------------------------
# 2. Set paths / options
# --------------------------------------------------
tweet_files = sorted(glob.glob("./datasets/tweet_*.json"))
output_csv = "./datasets/final_outputs/df_tweets_filtered.csv"
chunk_size = 50_000

Path(output_csv).parent.mkdir(parents=True, exist_ok=True)

print(f"Tweet files found: {len(tweet_files)}")

# --------------------------------------------------
# 3. Stream through raw tweet files and filter
# --------------------------------------------------
chunk = []
writer = None
total_seen = 0
total_matched = 0

with open(output_csv, "w", newline="", encoding="utf-8") as csv_file:
    for tweet_file in tweet_files:
        file_seen = 0
        file_matched = 0

        print(f"\nProcessing {tweet_file} ...")

        with open(tweet_file, "rb") as fin:
            try:
                for tweet in ijson.items(fin, "item", use_float=True):
                    file_seen += 1
                    total_seen += 1

                    # author_id is numeric/string like "12345"
                    # df_users_final["id"] is like "u12345"
                    author_id = tweet.get("author_id")
                    if author_id is None:
                        continue

                    uid = "u" + str(author_id)

                    # keep ALL tweets for sampled users
                    if uid not in user_ids:
                        continue

                    chunk.append(tweet)
                    file_matched += 1
                    total_matched += 1

                    if writer is None:
                        writer = DictWriter(
                            csv_file,
                            fieldnames=list(tweet.keys()),
                            extrasaction="ignore"
                        )
                        writer.writeheader()

                    if len(chunk) >= chunk_size:
                        writer.writerows(chunk)
                        chunk.clear()

            except ijson.common.IncompleteJSONError:
                print(f"{tweet_file} is incomplete/truncated — saving matched rows found so far")

        print(f"Seen: {file_seen:,} | Matched: {file_matched:,}")

    if chunk and writer:
        writer.writerows(chunk)
        chunk.clear()

print(f"\nDone.")
print(f"Total tweets scanned: {total_seen:,}")
print(f"Total tweets matched: {total_matched:,}")
print(f"Saved to: {output_csv}")

df_user_tweets_final = pd.read_csv("datasets/final_outputs/df_tweets_filtered.csv")
df_user_tweets_final.to_parquet("datasets/final_outputs/df_user_tweets_final.parquet", index=False)

Unique sampled user IDs: 20,000
Tweet files found: 9

Processing ./datasets\tweet_0 (1).json ...
Seen: 10,000,000 | Matched: 344,423

Processing ./datasets\tweet_1.json ...
Seen: 10,000,000 | Matched: 2,123,998

Processing ./datasets\tweet_2.json ...
Seen: 10,000,000 | Matched: 723,665

Processing ./datasets\tweet_3.json ...
Seen: 10,000,000 | Matched: 734,119

Processing ./datasets\tweet_4.json ...
Seen: 10,000,000 | Matched: 417,894

Processing ./datasets\tweet_5.json ...
Seen: 10,000,000 | Matched: 399,589

Processing ./datasets\tweet_6.json ...
Seen: 10,000,000 | Matched: 308,098

Processing ./datasets\tweet_7.json ...
Seen: 10,000,000 | Matched: 294,537

Processing ./datasets\tweet_8.json ...
Seen: 8,217,457 | Matched: 451,438

Done.
Total tweets scanned: 88,217,457
Total tweets matched: 5,797,761
Saved to: ./datasets/final_outputs/df_tweets_filtered.csv


# Analyse graph connectivity

In [ ]:
import pandas as pd
import networkx as nx

# Make sure IDs are strings
df_edges_final = pd.read_parquet(OUTPUT_EDGES_FINAL)
df_edges_final = df_edges_final.copy()
df_edges_final["source_id"] = df_edges_final["source_id"].astype(str)
df_edges_final["target_id"] = df_edges_final["target_id"].astype(str)
df_edges_final["relation"] = df_edges_final["relation"].astype(str)

# -------------------------------------------------
# 1. Basic counts
# -------------------------------------------------
all_nodes = set(df_edges_final["source_id"]) | set(df_edges_final["target_id"])

user_nodes = {n for n in all_nodes if n.startswith("u")}
tweet_nodes = {n for n in all_nodes if n.startswith("t")}
list_nodes = {n for n in all_nodes if n.startswith("l")}
hashtag_nodes = {n for n in all_nodes if n.startswith("h")}

print("=== BASIC COUNTS ===")
print("Total edges:", len(df_edges_final))
print("Total nodes:", len(all_nodes))
print("User nodes:", len(user_nodes))
print("Tweet nodes:", len(tweet_nodes))
print("List nodes:", len(list_nodes))
print("Hashtag nodes:", len(hashtag_nodes))
print()

# -------------------------------------------------
# 2. Relation counts
# -------------------------------------------------
print("=== RELATION COUNTS ===")
print(df_edges_final["relation"].value_counts())
print()

# -------------------------------------------------
# 3. Directed graph density
# -------------------------------------------------
# Density = m / (n * (n - 1)) for directed graph without self-loops
n = len(all_nodes)
m = len(df_edges_final)

density_directed = m / (n * (n - 1)) if n > 1 else 0.0

print("=== FULL DIRECTED GRAPH DENSITY ===")
print("Directed density:", density_directed)
print()

# -------------------------------------------------
# 4. Degree stats
# -------------------------------------------------
out_deg = df_edges_final.groupby("source_id").size()
in_deg = df_edges_final.groupby("target_id").size()

deg_df = pd.DataFrame(index=list(all_nodes))
deg_df["out_degree"] = out_deg
deg_df["in_degree"] = in_deg
deg_df = deg_df.fillna(0)
deg_df["total_degree"] = deg_df["out_degree"] + deg_df["in_degree"]

print("=== DEGREE STATS (ALL NODES) ===")
print(deg_df["total_degree"].describe())
print()

# -------------------------------------------------
# 5. User-only degree stats
# -------------------------------------------------
user_deg_df = deg_df.loc[deg_df.index.intersection(user_nodes)].copy()

print("=== DEGREE STATS (USER NODES ONLY) ===")
print(user_deg_df["total_degree"].describe())
print()

# -------------------------------------------------
# 6. User-user subgraph density
# -------------------------------------------------
df_user_user = df_edges_final[
    df_edges_final["source_id"].str.startswith("u") &
    df_edges_final["target_id"].str.startswith("u")
].copy()

uu_nodes = set(df_user_user["source_id"]) | set(df_user_user["target_id"])
uu_n = len(uu_nodes)
uu_m = len(df_user_user)

uu_density = uu_m / (uu_n * (uu_n - 1)) if uu_n > 1 else 0.0

print("=== USER-USER SUBGRAPH ===")
print("User-user edges:", uu_m)
print("User-user nodes:", uu_n)
print("User-user directed density:", uu_density)
print()

# -------------------------------------------------
# 7. Connectivity analysis
# -------------------------------------------------
G = nx.from_pandas_edgelist(
    df_edges_final,
    source="source_id",
    target="target_id",
    create_using=nx.DiGraph()
)

# Weakly connected components matter more for directed hetero graphs
weak_components = list(nx.weakly_connected_components(G))
weak_component_sizes = sorted([len(c) for c in weak_components], reverse=True)

largest_wcc = weak_component_sizes[0] if weak_component_sizes else 0
largest_wcc_ratio = largest_wcc / len(all_nodes) if len(all_nodes) > 0 else 0

print("=== CONNECTIVITY ===")
print("Number of weakly connected components:", len(weak_components))
print("Largest weakly connected component size:", largest_wcc)
print("Largest WCC ratio:", largest_wcc_ratio)
print("Top 10 component sizes:", weak_component_sizes[:10])
print()

# -------------------------------------------------
# 9. Isolates in final graph
# -------------------------------------------------
isolates = [node for node, degree in G.degree() if degree == 0]

print("=== ISOLATES ===")
print("Number of isolates:", len(isolates))
print("Isolate ratio:", len(isolates) / len(all_nodes) if len(all_nodes) else 0)
print()

relation
post          5797761
discuss       4970315
contain       1562941
mentioned      849521
retweeted      740414
replied_to     489842
like           203642
following      197990
quoted         135434
membership     102436
followers       32809
followed        17346
pinned           9120
own              2874
Name: count, dtype: int64
=== BASIC COUNTS ===
Total edges: 15112445
Total nodes: 8453949
User nodes: 19999
Tweet nodes: 7896467
List nodes: 16933
Hashtag nodes: 520550

=== RELATION COUNTS ===
relation
post          5797761
discuss       4970315
contain       1562941
mentioned      849521
retweeted      740414
replied_to     489842
like           203642
following      197990
quoted         135434
membership     102436
followers       32809
followed        17346
pinned           9120
own              2874
Name: count, dtype: int64

=== FULL DIRECTED GRAPH DENSITY ===
Directed density: 2.1145382113821428e-07

=== DEGREE STATS (ALL NODES) ===
count    8.453949e+06
mean     3.5